# Project Lab 2: Regression Detection, CI/CD Gates and Drift Monitoring

Notebook for **Chapter 16** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: Regression Detector Across Releases

In [ ]:
def bootstrap_ratio(a, b, B=2000, seed=0):
    r = np.random.default_rng(seed); ratios = np.empty(B)
    for i in range(B):
        ratios[i] = np.median(r.choice(b, len(b))) / np.median(r.choice(a, len(a)))
    return np.percentile(ratios, [2.5, 97.5])

In [ ]:
def verdict(a, b, min_effect=1.03, B=2000):
    lo, hi = bootstrap_ratio(a, b, B)
    if lo > min_effect:          return "REGRESSION", lo, hi
    if hi < 1 / min_effect:      return "improvement", lo, hi
    return "ok", lo, hi

In [ ]:
def cusum_changepoint(series):
    x = np.log(series); n = len(x); best, best_k = 0, None
    for k in range(2, n - 1):
        l, r = x[:k], x[k:]
        stat = abs(l.mean() - r.mean()) * np.sqrt(len(l) * len(r) / n)
        if stat > best: best, best_k = stat, k
    return best_k, best

change point: releases up to r8 vs from r9; stat=0.19, permutation p=0.0040


## 16.2 Project: CI/CD for ML With Model-Gating Tests

In [ ]:
RULES = {"max_mae_worsening_pct": 2.0, "max_p95_ms_worsening_pct": 10.0,
         "max_slice_mae_worsening_pct": 8.0, "min_rows": 500}

def pct_worse(new, old): return (new - old) / old * 100

def gate(candidate, baseline, rules=RULES):
    out = [("enough eval rows", candidate["eval_rows"] >= rules["min_rows"], "")]
    d = pct_worse(candidate["mae"], baseline["mae"])
    out.append(("overall MAE", d <= rules["max_mae_worsening_pct"], f"{d:+.2f}%"))
    d = pct_worse(candidate["p95_ms"], baseline["p95_ms"])
    out.append(("p95 latency", d <= rules["max_p95_ms_worsening_pct"], f"{d:+.2f}%"))
    for name, mae in candidate["slice_mae"].items():
        d = pct_worse(mae, baseline["slice_mae"][name])
        out.append((f"slice {name}", d <= rules["max_slice_mae_worsening_pct"], f"{d:+.2f}%"))
    return all(ok for _, ok, _ in out), out

## 16.3 Project: Drift Monitoring With a Retraining Trigger

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref); c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-4, None), np.clip(c, 1e-4, None)      # avoid log(0)
    return float(np.sum((c - r) * np.log(c / r)))

In [ ]:
def retrain_rule(history, psi_alert=0.25, features_needed=1, weeks_in_row=2):
    if len(history) < weeks_in_row: return False
    return all(sum(v >= psi_alert for v in w.values()) >= features_needed
               for w in history[-weeks_in_row:])

## Chapter recap

## Practice set

**1. (easy)** A bootstrap interval for the ratio new/old median is [0.97, 1.08]. Regression or not? **Answer:** Not proven. The interval includes 1.00 (and 0.97), so the data do not show a change. Collect more samples if the question matters.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch16.html